In [1]:
import json
from pathlib import Path
from collections import Counter

In [2]:
DATASET = Path("output/normalized_dataset.json")

with open(DATASET, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [3]:
print("========== DATASET OVERVIEW ==========\n")

print("Logs:", len(data["logs"]))
print("Uploads:", len(data["uploads"]))
print("Applications:", len(data["summary"]["identified_applications"]))
print("Users:", len(data["summary"]["identified_usernames"]))

========== DATASET OVERVIEW ==========

Logs: 2340
Uploads: 24956
Applications: 5
Users: 9


In [4]:
print("========== MISSING VALUES ==========\n")

missing = {}

for section in ["client","collection","requests"]:

    missing[section] = {}

    for key,value in data[section].items():

        if value is None:

            missing[section][key] = "Missing"

print(missing)

========== MISSING VALUES ==========

{'client': {}, 'collection': {}, 'requests': {}}


In [5]:
print("========== UPLOAD MISSING VALUES ==========\n")

missing_counter = Counter()

for upload in data["uploads"]:

    for key,value in upload.items():

        if value in [None,"",[]]:

            missing_counter[key]+=1

for key,value in sorted(missing_counter.items()):

    print(f"{key}: {value}")

========== UPLOAD MISSING VALUES ==========

application: 16540
extension: 6214
type: 24906
upload_percentage: 1611
username: 723


In [6]:
print("========== LOG MISSING VALUES ==========\n")

missing_counter = Counter()

for log in data["logs"]:

    for key,value in log.items():

        if value in [None,"",[]]:

            missing_counter[key]+=1

for key,value in sorted(missing_counter.items()):

    print(f"{key}: {value}")

========== LOG MISSING VALUES ==========

accessor: 1618
action: 1613
message: 3
target: 1613


In [7]:
print("========== DUPLICATE UPLOADS ==========\n")

seen=set()

duplicates=[]

for upload in data["uploads"]:

    key=(

        upload["path"],

        upload["timestamp"],

        upload["file_size"]

    )

    if key in seen:

        duplicates.append(upload)

    else:

        seen.add(key)

print("Duplicates:",len(duplicates))

========== DUPLICATE UPLOADS ==========

Duplicates: 0


In [8]:
print("========== DUPLICATE LOGS ==========\n")

seen=set()

duplicates=[]

for log in data["logs"]:

    key=(

        log["timestamp"],

        log["message"]

    )

    if key in seen:

        duplicates.append(log)

    else:

        seen.add(key)

print("Duplicates:",len(duplicates))

========== DUPLICATE LOGS ==========

Duplicates: 40


In [9]:
print("========== UPLOAD STATUS ==========\n")

complete=0

incomplete=0

unknown=0

for upload in data["uploads"]:

    status=upload["upload_complete"]

    if status==True:

        complete+=1

    elif status==False:

        incomplete+=1

    else:

        unknown+=1

print("Complete:",complete)
print("Incomplete:",incomplete)
print("Unknown:",unknown)

========== UPLOAD STATUS ==========

Complete: 24856
Incomplete: 100
Unknown: 0


In [10]:
print("========== ZERO BYTE FILES ==========\n")

zero=[]

for upload in data["uploads"]:

    if upload["file_size"]==0:

        zero.append(upload)

print("Zero byte files:",len(zero))

========== ZERO BYTE FILES ==========

Zero byte files: 1611


In [11]:
print("========== TIMESTAMP QUALITY ==========\n")

missing=0

for upload in data["uploads"]:

    if upload["timestamp"] is None:

        missing+=1

print("Uploads without timestamp:",missing)

missing=0

for log in data["logs"]:

    if log["timestamp"] is None:

        missing+=1

print("Logs without timestamp:",missing)

========== TIMESTAMP QUALITY ==========

Uploads without timestamp: 0
Logs without timestamp: 0


In [12]:
print("========== USERNAMES ==========\n")

counter=Counter()

for upload in data["uploads"]:

    username=upload["username"]

    if username:

        counter[username]+=1

print(counter)

========== USERNAMES ==========

Counter({'yscott': 15738, 'wayne': 3484, 'it01': 2189, 'Abdallah.Kh': 1418, 'dallen': 1180, 'administrator': 91, 'defaultuser0': 75, 'Default': 54, 'Public': 4})


In [13]:
print("========== APPLICATIONS ==========\n")

counter=Counter()

for upload in data["uploads"]:

    app=upload["application"]

    if app:

        counter[app]+=1

print(counter)

========== APPLICATIONS ==========

Counter({'Microsoft Edge': 4689, 'OneDrive': 3545, 'Internet Explorer': 122, 'Windows WebCache': 49, 'PowerShell': 11})


In [14]:
print("========== FILE EXTENSIONS ==========\n")

counter=Counter()

for upload in data["uploads"]:

    ext=upload["extension"]

    counter[ext]+=1

print(counter.most_common(30))

========== FILE EXTENSIONS ==========

[('', 6214), ('.js', 2823), ('.vim', 1853), ('.json', 1600), ('.adoc', 835), ('.dll', 754), ('.dat', 721), ('.svg', 718), ('.pm', 640), ('.txt', 568), ('.exe', 448), ('.png', 442), ('.html', 422), ('.log', 382), ('.lock', 372), ('.lnk', 347), ('.odlgz', 340), ('.mui', 297), ('.odl', 273), ('.db', 248), ('.pf', 221), ('.hyb', 208), ('.ini', 192), ('.tcl', 173), ('.evtx', 158), ('.msg', 149), ('.old', 143), ('.ts', 131), ('.log2', 126), ('.log1', 126)]


In [15]:
print("========== SUSPICIOUS UPLOADS ==========\n")

suspicious=[]

for upload in data["uploads"]:

    if upload["upload_complete"]==False:

        suspicious.append(upload)

print("Suspicious uploads:",len(suspicious))

========== SUSPICIOUS UPLOADS ==========

Suspicious uploads: 100


In [16]:
print("========== FLAGGING DUPLICATE UPLOADS ==========\n")

seen = set()
duplicate_count = 0

for upload in data["uploads"]:

    key = (
        upload["path"],
        upload["timestamp"],
        upload["file_size"]
    )

    if key in seen:
        upload["is_duplicate"] = True
        duplicate_count += 1
    else:
        upload["is_duplicate"] = False
        seen.add(key)

print("Duplicate uploads found:", duplicate_count)

========== FLAGGING DUPLICATE UPLOADS ==========

Duplicate uploads found: 0


In [17]:
print("========== FLAGGING DUPLICATE LOGS ==========\n")

seen = set()
duplicate_count = 0

for log in data["logs"]:

    key = (
        log["timestamp"],
        log["message"]
    )

    if key in seen:
        log["is_duplicate"] = True
        duplicate_count += 1
    else:
        log["is_duplicate"] = False
        seen.add(key)

print("Duplicate logs found:", duplicate_count)

========== FLAGGING DUPLICATE LOGS ==========

Duplicate logs found: 40


In [18]:
print("========== ADDING FORENSIC FLAGS ==========\n")

for upload in data["uploads"]:

    # Zero-byte file
    upload["zero_byte"] = upload["file_size"] == 0

    # Username exists
    upload["has_username"] = upload["username"] is not None

    # Application identified
    upload["has_application"] = upload["application"] is not None

print("Forensic flags added.")

========== ADDING FORENSIC FLAGS ==========

Forensic flags added.


In [19]:
print("========== BUILDING ANOMALY SUMMARY ==========\n")

anomalies = {

    "duplicate_uploads": sum(
        1 for u in data["uploads"]
        if u["is_duplicate"]
    ),

    "duplicate_logs": sum(
        1 for l in data["logs"]
        if l["is_duplicate"]
    ),

    "incomplete_uploads": sum(
        1 for u in data["uploads"]
        if not u["upload_complete"]
    ),

    "zero_byte_files": sum(
        1 for u in data["uploads"]
        if u["zero_byte"]
    ),

    "uploads_without_username": sum(
        1 for u in data["uploads"]
        if not u["has_username"]
    ),

    "uploads_without_application": sum(
        1 for u in data["uploads"]
        if not u["has_application"]
    )

}

print(json.dumps(anomalies, indent=2))

========== BUILDING ANOMALY SUMMARY ==========

{
  "duplicate_uploads": 0,
  "duplicate_logs": 40,
  "incomplete_uploads": 100,
  "zero_byte_files": 1611,
  "uploads_without_username": 723,
  "uploads_without_application": 16540
}


In [20]:
processed_dataset = {

    "dataset_metadata": data["dataset_metadata"],

    "summary": data["summary"],

    "anomalies": anomalies,

    "client": data["client"],

    "collection": data["collection"],

    "requests": data["requests"],

    "logs": data["logs"],

    "uploads": data["uploads"]

}

In [21]:
OUTPUT=Path("output/processed_dataset.json")

with open(OUTPUT,"w",encoding="utf-8") as f:

    json.dump(

        processed_dataset,

        f,

        indent=2,

        ensure_ascii=False

    )

print("Processed dataset saved.")

Processed dataset saved.
